In [1]:
# imports
import pandas as pd
import numpy as np
import random
import time
from collections import Counter
from ingredient_normalisation import normalise_user_ingredients
from evaluation_queries import make_random_queries, MIN_SOURCE_INGREDIENTS
from baseline_retrieval import jaccard_model, recipes_dataset_normalised

# frozen final settings - selected using validation results
# random seeds
split_seed = 34
# evaluation settings
# retrieve top 100 candidates before reranking
candidate_k = 100
# compare recommendations at top 5 and top 10
evaluation_k = [5, 10]
query_lengths = [3, 4, 5, 6]
queries_per_run = 50
number_of_runs = 30

# use new fixed seeds for held-out test queries
test_seeds = list(range(1000, 1000 + number_of_runs))

# check number of recipes and columns loaded
print("dataset shape:", recipes_dataset_normalised.shape)
print("test seeds:", test_seeds)


c:\Users\preen\Documents\AIML339 - FINAL\AIML339 - FINAL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dataset shape: (10000, 6)
test seeds: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029]


In [2]:
# CREATE VALIDATION AND TEST SET

# use the same eligible source recipes and split as development
# select recipes with at least 7 different ingredients
valid_recipes = recipes_dataset_normalised[recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: len(set(ingredients)) >= MIN_SOURCE_INGREDIENTS)]

# shuffle the source recipe indices using chosen seed 34
recipe_indices = valid_recipes.index.tolist()
# shuffle using split seed so validation and test recipes stay the same each run
random.Random(split_seed).shuffle(recipe_indices)

# split source recipes into separate validation and test halves
split_point = len(recipe_indices) // 2
validation_indices = recipe_indices[:split_point]
test_indices = recipe_indices[split_point:]

# check that validation and test source recipes do not overlap
assert set(validation_indices).isdisjoint(test_indices)
assert len(test_indices) >= queries_per_run
print("test source recipes:", len(test_indices))


test source recipes: 4911


In [ ]:
# GENERATE TEST QUERIES

# use the same query generation function as the validation experiments
def generate_queries(seed, partition="test"):
    # generate 50 source recipes with nested queries of 3, 4, 5 and 6 ingredients
    return make_random_queries(recipes_dataset_normalised, partition=partition, query_lengths=query_lengths, number_of_queries=queries_per_run, split_seed=split_seed, query_seed=seed, min_source_ingredients=MIN_SOURCE_INGREDIENTS)


In [4]:
# EVALUATION - RELEVANCE

#assign relevance score based on matched & missing ingredients
def recipe_relevance(user_ingredients, recipe_ingredients):

    # convert ingredients from user and recipe to sets
    user_set = set(user_ingredients)
    recipe_set = set(recipe_ingredients)

    # calculate matched and missing ingredients
    matched = len(user_set & recipe_set)
    missing = len(recipe_set - user_set)

    # no ingredients matched
    if matched == 0:
        return 0

    # some ingredients matched
    if matched < len(user_set):
        return 1

    # all ingredients matched, with few missing
    if missing <= 2:
        return 3

    # all ingredients matched, but more are missing
    return 2

In [5]:
# CCALCULATE RELEVANCE

# relecance for each recipe using user query
def prepare_query_relevance(user_input):
    #normalise ingredients and convert to set
    user_ingredients = normalise_user_ingredients(user_input)
    # convert ingredients from user and recipe to sets
    user_set = set(user_ingredients)

    # relevance score assigned to each recipe - mark recipes containing all ingredients as relevant
    relevance_scores = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: recipe_relevance(user_ingredients, ingredients))
    binary_relevance = recipes_dataset_normalised["processing_ingredients"].apply(lambda ingredients: user_set.issubset(set(ingredients)))
    # total no. of relevant recipes
    total_relevant = int(binary_relevance.sum())

    return relevance_scores, binary_relevance, total_relevant


In [6]:
#PRECISION, RECALL, NDGC, COVERAGE + MISSING INGREDIENTS

def evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant):
    # select top k
    top_recipes = recommendations.head(k)
    user_set = set(normalise_user_ingredients(user_input))
    #calculate found relevant in topk
    relevant_found = int(binary_relevance.loc[top_recipes.index].sum())
    # proportion of top k recipes containing all user ingredients
    precision = relevant_found / k
    # proportion of all relevant recipes retrieved - undefined if none are relevant
    recall = relevant_found / total_relevant if total_relevant > 0 else np.nan

    # calculate scores in recommendation orders
    grades = relevance_scores.loc[top_recipes.index].to_numpy()
    #so lower ranked recipes contribute less
    discounts = np.log2(np.arange(2, len(grades) + 2))
    # using relevance scores and rank discounts
    dcg = np.sum((2 ** grades - 1) / discounts)

    #
    # select highest relevance scores in dataset for ideal ranking
    ideal_grades = np.sort(relevance_scores.to_numpy())[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal_grades) + 2))
    ideal_dcg = np.sum((2 ** ideal_grades - 1) / ideal_discounts)
    # compare recommendation ranking with ideal ranking
    ndcg = dcg / ideal_dcg if ideal_dcg > 0 else np.nan

    # empty list to add
    coverages = []
    missing_counts = []

    # calculate coverage and missing ingredients for each recommended recipe
    for ingredients in top_recipes["processing_ingredients"]:
        recipe_set = set(ingredients)
        # proportion of user ingredients found in recipe
        coverages.append(len(user_set & recipe_set) / len(user_set))
        # count recipe ingredients user doesnt have
        missing_counts.append(len(recipe_set - user_set))

    return {"precision": precision, "recall": recall, "ndcg": ndcg, "coverage": np.mean(coverages), "missing": np.mean(missing_counts)}


In [7]:
# COMBINED RERANKING V1 AND V2

# calculate how frequently each ingredient occurs in the recipe dataset

ingredient_counts = Counter()

# go through ingredients of each recipe
for ingredients in recipes_dataset_normalised["processing_ingredients"]:
    # count each ingredient once per recipe
    ingredient_counts.update(set(ingredients))

# proportion of recipes containing each ingredient
ingredient_frequencies = {ingredient: count / len(recipes_dataset_normalised) for ingredient, count in ingredient_counts.items()}

# calculate features for each candidate recipe
def prepare_reranking_features(candidates, user_input, similarity_column):
    # candidate recipes copy to store new reranking scores
    reranked = candidates.copy()
    user_set = set(normalise_user_ingredients(user_input))
    # scale similarity scores between 0 and 1
    scores = reranked[similarity_column]
    #lowest and highest
    minimum = scores.min()
    maximum = scores.max()

    # min-max scaling - use 0 if all candidate similarities are equal
    reranked["scaled_similarity"] = 0.0 if maximum == minimum else (scores - minimum) / (maximum - minimum)
    #proportion of user query in recipe
    reranked["coverage"] = reranked["processing_ingredients"].apply(lambda ingredients: len(user_set & set(ingredients)) / len(user_set))
    # proportion of recipe ingredients user doesnt have - original v1 penalty
    reranked["missing_penalty_v1"] = reranked["processing_ingredients"].apply(lambda ingredients: len(set(ingredients) - user_set) / len(set(ingredients)) if ingredients else 0)
    #ingredients user doesnt have
    reranked["missing_count"] = reranked["processing_ingredients"].apply(lambda ingredients: len(set(ingredients) - user_set))
    # reward common missing ingredients
    reranked["common_reward"] = reranked["processing_ingredients"].apply(lambda ingredients: sum(ingredient_frequencies.get(i, 0) for i in set(ingredients) - user_set) / len(set(ingredients)) if ingredients else 0)

    # return candidate recipes with calculated reranking features
    return reranked

# rerank
def rerank_configuration(features, coverage_weight=0.5, missing_weight=0.4, common_weight=0.1, missing_scale=5, missing_method="v2"):
    # copy features so each configuration keeps its own scores
    reranked = features.copy()
    # use original v1 proportion or v2 missing count - cap v2 penalty at 1
    missing_penalty = reranked["missing_penalty_v1"] if missing_method == "v1" else (reranked["missing_count"] / missing_scale).clip(upper=1)
    # calculate final score
    reranked["final_score"] = reranked["scaled_similarity"] + coverage_weight * reranked["coverage"] - missing_weight * missing_penalty + common_weight * reranked["common_reward"]
    # return in order
    # return candidate recipes with calculated reranking features
    return reranked.sort_values("final_score", ascending=False)

## Frozen final configurations

Comparing the Jaccard Baseline vs Jaccard + Combined V2 



In [8]:
# FINAL TEST CONFIGURATIONS

# compare original jaccard retrieval against combined v2
# original v1 and v2 weights, test each feature
configurations = {
    "Baseline": ("baseline", {}),
    "Combined V2": ("v2", {"coverage_weight": 0.5, "missing_weight": 0.4, "common_weight": 0.1, "missing_scale": 5, "missing_method": "v2"})
}

print("configurations:", list(configurations))


configurations: ['Baseline', 'Combined V2']


In [9]:
# RUNTIME COMPARISON - JACCARD AND COMBINED V2

# use validation queries to measure runtime separately from held-out test
runtime_queries = generate_queries(84, partition="validation")
runtime_repeats = 3
runtime_results = []

# warm up retrieval before timing
for user_input in runtime_queries["query"].head(3):
    jaccard_model(user_input, candidate_k)

# go through every query and repeat measurements
for query_number, query_row in runtime_queries.iterrows():
    user_input = query_row["query"]
    for repeat in range(runtime_repeats):
        # time retrieval of 100 candidates
        start = time.perf_counter()
        candidates = jaccard_model(user_input, candidate_k)
        # convert elapsed retrieval time from seconds to milliseconds
        retrieval_ms = (time.perf_counter() - start) * 1000

        # time shared feature calculation
        start = time.perf_counter()
        features = prepare_reranking_features(candidates, user_input, "jaccard_similarity")
        feature_ms = (time.perf_counter() - start) * 1000

        # compare baseline and combined v2 using the same candidates
        for configuration_name, (method, settings) in configurations.items():
            start = time.perf_counter()
            if method == "baseline":
                recommendations = candidates
            else:
                recommendations = rerank_configuration(features, **settings)
            # record reranking time for combined v2 - baseline has no reranking step
            rerank_ms = (time.perf_counter() - start) * 1000 if method != "baseline" else 0.0

            # store retrieval, feature and reranking times for each query and repeat
            runtime_results.append({"query_id": query_row["query_id"], "repeat": repeat + 1,
                                    "configuration": configuration_name,
                                    "retrieval_ms": retrieval_ms,
                                    "feature_ms": feature_ms if method != "baseline" else 0.0,
                                    "rerank_ms": rerank_ms,
                                    "total_ms": retrieval_ms + (feature_ms + rerank_ms if method != "baseline" else 0.0)})

# save individual timings and average time for each configuration
final_test_runtime_results = pd.DataFrame(runtime_results)
# average timings across queries and repeats for each configuration
final_test_runtime_summary = final_test_runtime_results.groupby("configuration", as_index=False).agg(
    retrieval_mean_ms=("retrieval_ms", "mean"), feature_mean_ms=("feature_ms", "mean"),
    rerank_mean_ms=("rerank_ms", "mean"), total_mean_ms=("total_ms", "mean"),
    total_std_ms=("total_ms", "std"))
final_test_runtime_results.to_csv("final_test_runtime_results.csv", index=False)
final_test_runtime_summary.to_csv("final_test_runtime_summary.csv", index=False)
display(final_test_runtime_summary)


,configuration,retrieval_mean_ms,feature_mean_ms,rerank_mean_ms,total_mean_ms,total_std_ms
0,Baseline,42.359832,0.000000,0.000000,42.359832,35.677992
1,Combined V2,42.359832,6.184875,4.234255,52.778962,42.135932


## Run held-out evaluation

Jaccard baseline vs Combined V2 on the same queries and candidates. AFTER freezing all settings.


In [10]:
# EVALUATE 30 TEST QUERY SAMPLES
results = []

# go through each test seed
for run_number, test_seed in enumerate(test_seeds, start=1):
    # generate new repeatable test queries for this seed
    test_query_set = generate_queries(test_seed, partition="test")

    # go through every query
    for query_number, query_row in test_query_set.iterrows():
        user_input = query_row["query"]
        # calculate relevance across dataset once for this query
        relevance_scores, binary_relevance, total_relevant = prepare_query_relevance(user_input)

        # retrieve the same 100 candidates for both configurations
        baseline_candidates = jaccard_model(user_input, candidate_k)
        # calculate features once for same candidates used by baseline and combined v2
        features = prepare_reranking_features(baseline_candidates, user_input, "jaccard_similarity")

        # evaluate original baseline and combined v2
        for configuration_name, (method, settings) in configurations.items():
            if method == "baseline":
                recommendations = baseline_candidates
            else:
                recommendations = rerank_configuration(features, **settings)

            for k in evaluation_k:
                metrics = evaluate_recommendations(recommendations, user_input, k, relevance_scores, binary_relevance, total_relevant)
                # save query details, test seed, configuration and calculated metrics
                results.append({"seed": test_seed, "query_id": query_row["query_id"],
                                "source_recipe": query_row["source_index"], "query_length": query_row["query_length"],
                                "query": user_input, "model": "Jaccard", "configuration": configuration_name,
                                "candidate_k": candidate_k, "k": k, "total_relevant": total_relevant, **metrics})

    print(f"completed run {run_number}/{number_of_runs} (seed {test_seed})")

# convert results to dataframe
final_test_results = pd.DataFrame(results)
# expected rows from runs, source recipes, query lengths, configurations and top k
expected_rows = number_of_runs * queries_per_run * len(query_lengths) * len(configurations) * len(evaluation_k)
# check every test query and configuration produced evaluation results
assert len(final_test_results) == expected_rows
print("result rows:", len(final_test_results))


completed run 1/30 (seed 1000)
completed run 2/30 (seed 1001)
completed run 3/30 (seed 1002)
completed run 4/30 (seed 1003)
completed run 5/30 (seed 1004)
completed run 6/30 (seed 1005)
completed run 7/30 (seed 1006)
completed run 8/30 (seed 1007)
completed run 9/30 (seed 1008)
completed run 10/30 (seed 1009)
completed run 11/30 (seed 1010)
completed run 12/30 (seed 1011)
completed run 13/30 (seed 1012)
completed run 14/30 (seed 1013)
completed run 15/30 (seed 1014)
completed run 16/30 (seed 1015)
completed run 17/30 (seed 1016)
completed run 18/30 (seed 1017)
completed run 19/30 (seed 1018)
completed run 20/30 (seed 1019)
completed run 21/30 (seed 1020)
completed run 22/30 (seed 1021)
completed run 23/30 (seed 1022)
completed run 24/30 (seed 1023)
completed run 25/30 (seed 1024)
completed run 26/30 (seed 1025)
completed run 27/30 (seed 1026)
completed run 28/30 (seed 1027)
completed run 29/30 (seed 1028)
completed run 30/30 (seed 1029)
result rows: 24000


## Summarise results

Average over the 50 queries within each seed, then report mean ± standard deviation across the 30 seeds.

In [11]:
# SUMMARISE FINAL TEST RESULTS
# metrics to average across queries and test seeds
metric_columns = ["precision", "recall", "ndcg", "coverage", "missing"]

# average metrics over queries within each seed
final_test_per_seed = final_test_results.groupby(["seed", "model", "configuration", "k"], as_index=False)[metric_columns].mean()

# calculate mean and standard deviation across 30 seeds
final_test_summary = final_test_per_seed.groupby(["model", "configuration", "k"])[metric_columns].agg(["mean", "std"]).round(4)

display(final_test_summary)


precision          recall            ndcg          \
                              mean     std    mean     std    mean     std   
model   configuration k                                                      
Jaccard Baseline      5     0.2487  0.0212  0.7611  0.0342  0.8858  0.0149   
                      10    0.1598  0.0201  0.8090  0.0312  0.9048  0.0119   
        Combined V2   5     0.2560  0.0216  0.7752  0.0337  0.8922  0.0150   
                      10    0.1647  0.0207  0.8239  0.0295  0.9101  0.0119   

                         coverage         missing          
                             mean     std    mean     std  
model   configuration k                                    
Jaccard Baseline      5    0.6827  0.0142  3.1556  0.1485  
                      10   0.6275  0.0167  3.3860  0.1322  
        Combined V2   5    0.6664  0.0154  2.8958  0.1569  
                      10   0.6082  0.0178  3.0711  0.1319

## Paired differences against baseline

Compare Combined V2 with the Jaccard baseline using the **same test queries and seeds**. Negative differences for missing ingredients mean V2 requires fewer additional ingredients. These are descriptive paired comparisons.


In [12]:
# PAIRED DIFFERENCES FROM BASELINE
# index baseline averages by seed and top k for paired comparison
baseline_seed_scores = final_test_per_seed[final_test_per_seed["configuration"] == "Baseline"].set_index(["seed", "k"])
paired_rows = []

# compare every reranker with the same-seed baseline
for _, row in final_test_per_seed[final_test_per_seed["configuration"] != "Baseline"].iterrows():
    # select baseline using same test seed and top k
    baseline_row = baseline_seed_scores.loc[(row["seed"], row["k"])]
    # subtract baseline from reranker metrics - negative missing difference means fewer extras
    differences = {f"{metric}_difference": row[metric] - baseline_row[metric] for metric in metric_columns}
    paired_rows.append({"seed": row["seed"], "model": row["model"],
                        "configuration": row["configuration"], "k": row["k"], **differences})

final_test_paired_differences = pd.DataFrame(paired_rows)
# calculate mean and std of paired differences across test seeds
final_test_paired_summary = final_test_paired_differences.groupby(["model", "configuration", "k"]).agg(["mean", "std"]).round(4)
display(final_test_paired_summary)


seed         precision_difference          \
                            mean     std                 mean     std   
model   configuration k                                                 
Jaccard Combined V2   5   1014.5  8.8034               0.0074  0.0042   
                      10  1014.5  8.8034               0.0049  0.0034   

                         recall_difference         ndcg_difference          \
                                      mean     std            mean     std   
model   configuration k                                                      
Jaccard Combined V2   5             0.0141  0.0073          0.0063  0.0032   
                      10            0.0149  0.0096          0.0053  0.0027   

                         coverage_difference         missing_difference  \
                                        mean     std               mean   
model   configuration k                                                   
Jaccard Combined V2   5              -0.0163  0.0039            -0.2598   
                      10             -0.0193  0.0031            -0.3149   

                                  
                             std  
model   configuration k           
Jaccard Combined V2   5   0.0624  
                      10  0.0534

In [13]:
# SAVE FINAL TEST EVALUATION RESULTS
# save detailed query results, per-seed averages and overall summary
final_test_results.to_csv("final_test_results.csv", index=False)
final_test_per_seed.to_csv("final_test_per_seed.csv", index=False)
final_test_summary.to_csv("final_test_summary.csv")
# save paired differences and summary for comparison with baseline
final_test_paired_differences.to_csv("final_test_paired_differences.csv", index=False)
final_test_paired_summary.to_csv("final_test_paired_summary.csv")
